# CareLoop Clinics — No-Show Guard
## Part 1: Python & Data Foundations

Run this notebook from top to bottom. Keep the original source files in `data/`:
- `appointments.csv`
- `clinics.csv`
- `followup_notes.csv`

The generated hand-off files are written to the project root.


In [1]:
import csv
import json

def parse_row(row):
    try:
        parsed = dict(row)
        parsed['age'] = int(row['age'])
        parsed['fee_inr'] = int(row['fee_inr'])
        parsed['prior_no_shows'] = int(row['prior_no_shows'])
        parsed['distance_km'] = float(row['distance_km'])
        return parsed
    except ValueError:
        return None

def ingest_appointments(file_path):
    accepted_rows = []
    rejected_rows = []

    with open(file_path, 'r', newline='', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        headers = reader.fieldnames
        print(f"Headers: {headers}")

        for row in reader:
            parsed_row = parse_row(row)
            if parsed_row is not None:
                accepted_rows.append(parsed_row)
            else:
                rejected_rows.append(row)

    print(f"Total Rows Processed: {len(accepted_rows) + len(rejected_rows)}")
    print(f"Accepted Rows: {len(accepted_rows)} (Target: 988)")
    print(f"Rejected Rows: {len(rejected_rows)} (Target: 32)")

    with open('rejected_rows.json', 'w', encoding='utf-8') as json_file:
        json.dump(rejected_rows, json_file, indent=2)

    return accepted_rows

In [2]:
import pandas as pd

def clean_and_parse_data(file_path):
    df = pd.read_csv(file_path)
    print(f"\nTask 1.2 - Initial shape: {df.shape} (Target: (1020, 14))")
    print("\n--- Info ---")
    df.info()
    print("\n--- Null Counts ---")
    print(df.isnull().sum())
    print(f"\nExact duplicates found: {df.duplicated().sum()} (Target: 20)")

    initial_count = len(df)


    df = df.drop_duplicates()
    print(f"Task 1.2 (a) - Dropped {initial_count - len(df)} duplicate rows | Remaining: {len(df)} (Target: 1000)")


    df['booking_channel'] = df['booking_channel'].str.strip().str.title()
    print(f"Task 1.2 (b) - Unique channels: {list(df['booking_channel'].unique())} | Remaining: {len(df)}")


    df['appt_date'] = pd.to_datetime(df['appt_date'], errors='coerce')
    df['booked_on'] = pd.to_datetime(df['booked_on'])
    print(f"Task 1.2 (c) - Dates converted | Remaining: {len(df)}")


    df = df.dropna(subset=['appt_date', 'status'])
    print(f"Task 1.2 (d) - Shape after dropping missing appt_date/status: {df.shape} (Target: (990, 14))")

    return df

In [3]:
import numpy as np

def handle_validity_and_outlier(df):
    df = df.copy()


    invalid_age_mask = (df['age'] < 1) | (df['age'] > 100)
    print(f"\nTask 1.3 - Invalid ages found: {invalid_age_mask.sum()} (Target: 5)")
    df.loc[invalid_age_mask, 'age'] = np.nan

    missing_age_count = df['age'].isna().sum()
    print(f"Task 1.3 - Total missing ages to impute: {missing_age_count} (Target: 17)")


    df['age'] = df['age'].fillna(df['age'].median()).round().astype(int)
    df['distance_km'] = df['distance_km'].fillna(df['distance_km'].median())


    for col in ['distance_km', 'fee_inr']:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_cap = Q1 - 1.5 * IQR
        upper_cap = Q3 + 1.5 * IQR

        capped_count = ((df[col] < lower_cap) | (df[col] > upper_cap)).sum()
        df[col] = np.clip(df[col], lower_cap, upper_cap)
        print(f"Task 1.3 - {col}: Q1={Q1}, Q3={Q3}, Upper={upper_cap} | Capped={capped_count}")

    df['distance_km'] = df['distance_km'].round(1)
    df['fee_inr'] = df['fee_inr'].round().astype(int)

    return df

In [4]:
import numpy as np
def engineer_features_and_summarize(df):
  df['lead_days'] = (df['appt_date'] - df['booked_on']).dt.days
  df['no_show'] = np.where(df['status'] == 'No-Show' , 1, 0)
  print(f"\nTask 1.4 - lead_days range: {df['lead_days'].min()} to {df['lead_days'].max()} (Target: 1 to 45)")
  print(f"Task 1.4 - Total no-shows: {df['no_show'].sum()} / {len(df)} ({df['no_show'].mean() * 100:.1f}%) (Target: 218 / 22.0%)")

  X = df[['age', 'distance_km', 'fee_inr', 'lead_days']].to_numpy()

  col_means = np.mean(X, axis=0)
  print(f"Task 1.4 (a) - Column means [age, distance_km, fee_inr, lead_days]: {np.round(col_means, 2)}")
  print("               (Targets: [48.17, 9.50, 635.71, 11.82])")

  lead_mask = df['lead_days'].to_numpy()>14
  print(f"Task 1.4 (b) - Rows with lead_days > 14: {lead_mask.sum()} ({lead_mask.mean() * 100:.1f}%) (Target: 296 / 29.9%)")


  dist_arr = df['distance_km'].to_numpy()
  z_score = (dist_arr - np.mean(dist_arr)) / np.std(dist_arr)
  extreme_z_count = (np.abs(z_score) > 2).sum()
  print(f"Task 1.4 (c) - Rows with |z| > 2 for distance_km: {extreme_z_count} (Target: 53)")

  return df

In [5]:
import requests
import pandas as pd

def fetch_single_clinic_temp(clinic_name , latitude , longitude):
  url = "https://api.open-meteo.com/v1/forecast"
  params = {
      "latitude": latitude,
      "longitude": longitude,
      "hourly": "temperature_2m",
      "forecast_days": 1
  }
  failure_reason = "Unknown Error"


  for attempt in range(2):
    try:
      response = requests.get(url, params=params, timeout = 10)
      if response.status_code == 200:
        data = response.json()
        hourly = data.get("hourly")
        if hourly and "temperature_2m" in hourly and len(hourly["temperature_2m"]) > 0:
          return hourly["temperature_2m"][0]
        failure_reason = "Missing 'hourly' or 'temperature_2m' in API response"
      else:
        failure_reason = f"HTTP status {response.status_code}"
    except requests.exceptions.RequestException as err:
      failure_reason = str(err)

  print(f"Task 1.5 - Warning: Could not fetch temperature for '{clinic_name}' ({failure_reason})")
  return None

def enrich_clinics(clinics_path, output_csv="clinics_enriched.csv"):
    clinics_df = pd.read_csv(clinics_path)
    temps = []

    for _, row in clinics_df.iterrows():
        temp = fetch_single_clinic_temp(row['clinic_name'], row['latitude'], row['longitude'])
        temps.append(temp)

    clinics_df['temp_c'] = temps
    clinics_df.to_csv(output_csv, index=False)
    print(f"\nTask 1.5 - Saved {output_csv} ({len(clinics_df)} rows):")
    print(clinics_df[['clinic_id', 'clinic_name', 'city', 'temp_c']])

    return clinics_df


In [6]:
def export_clean_sql(df, clinics_df, clean_csv="appointments_clean.csv" , sql_path="load_data.sql"):
  df_out = df.sort_values('appointment_id').copy()
  df_out['booked_on']=df_out['booked_on'].dt.strftime('%Y-%m-%d')
  df_out['appt_date']=df_out['appt_date'].dt.strftime('%Y-%m-%d')

  ordered_cols = [
        'appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date',
        'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel',
        'visit_type', 'reminder_sent', 'prior_no_shows', 'status',
        'lead_days', 'no_show'
    ]

  df_out = df_out[ordered_cols]
  unique_appts = df_out['appointment_id'].nunique()
  total_missing = df_out.isnull().sum().sum()

  print(f"\nTask 1.6 - Unique appointment_id count: {unique_appts} (Target: 990)")
  print(f"Task 1.6 - Missing values across 16 columns: {total_missing} (Target: 0)")

  assert unique_appts == len(df_out),"Error: appointment_id is not unique!"
  assert total_missing == 0 ,"Error: Found missing values in export DataFrame!"

  df_out.to_csv(clean_csv, index=False)
  print(f"Task 1.6 - Exported {clean_csv} ({df_out.shape})")

  sql_statements = []

  for _, r in clinics_df.iterrows():
        temp_sql = "NULL" if pd.isna(r['temp_c']) else f"{r['temp_c']}"
        stmt = (
            f"INSERT INTO clinics VALUES ("
            f"'{r['clinic_id']}','{r['clinic_name']}','{r['city']}','{r['tier']}',"
            f"{r['latitude']},{r['longitude']},{temp_sql});"
        )
        sql_statements.append(stmt)

  for _ , r in df_out.iterrows():
    stmt = (
        f"INSERT INTO appointments VALUES ("
        f"'{r['appointment_id']}','{r['patient_id']}','{r['clinic_id']}',"
        f"'{r['booked_on']}','{r['appt_date']}',{r['age']},'{r['gender']}',"
        f"{r['distance_km']},{r['fee_inr']},'{r['booking_channel']}',"
        f"'{r['visit_type']}','{r['reminder_sent']}',{r['prior_no_shows']},"
        f"'{r['status']}',{r['lead_days']},{r['no_show']});"
    )
    sql_statements.append(stmt)

  with open(sql_path, 'w' , encoding = 'utf-8') as file:
    file.write('\n'.join(sql_statements) + "\n")

  print(f"Task 1.6 - Wrote {len(sql_statements)} INSERT statements to {sql_path} (Target: 998)")
  return df_out

In [7]:
accepted_rows = ingest_appointments("data/appointments.csv")
df_step1 = clean_and_parse_data("data/appointments.csv")
df_step2 = handle_validity_and_outlier(df_step1)
df_step3 = engineer_features_and_summarize(df_step2)
clinics_df = enrich_clinics("data/clinics.csv")
df_clean = export_clean_sql(df_step3, clinics_df)

Headers: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total Rows Processed: 1020
Accepted Rows: 988 (Target: 988)
Rejected Rows: 32 (Target: 32)

Task 1.2 - Initial shape: (1020, 14) (Target: (1020, 14))

--- Info ---
<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   str    
 1   patient_id       1020 non-null   str    
 2   clinic_id        1020 non-null   str    
 3   booked_on        1020 non-null   str    
 4   appt_date        1019 non-null   str    
 5   age              1008 non-null   float64
 6   gender           1020 non-null   str    
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-n


Task 1.5 - Saved clinics_enriched.csv (8 rows):
  clinic_id               clinic_name     city  temp_c
0       C01     Kothrud Family Clinic     Pune    18.3
1       C02      Hadapsar Care Centre     Pune    20.0
2       C03    Sitabuldi Health Point   Nagpur    20.4
3       C04      Malviya Nagar Clinic   Jaipur    16.2
4       C05     Vaishali Nagar Clinic   Jaipur    16.2
5       C06  Edappally Medical Centre    Kochi    26.2
6       C07        Vijay Nagar Clinic   Indore    16.6
7       C08    Gomti Nagar Care Point  Lucknow    16.9

Task 1.6 - Unique appointment_id count: 990 (Target: 990)
Task 1.6 - Missing values across 16 columns: 0 (Target: 0)
Task 1.6 - Exported appointments_clean.csv ((990, 16))


Task 1.6 - Wrote 998 INSERT statements to load_data.sql (Target: 998)
